# BÀI THỰC HÀNH CHƯƠNG 6 — TEXT CLUSTERING & TOPIC MODELLING

**Môn:** Xử lý ngôn ngữ tự nhiên  
**Hình thức:** Bài tập về nhà  
**Dữ liệu:** Tin tức hoặc phản hồi khách hàng tiếng Việt

## Mục tiêu
Sinh viên xây dựng và so sánh 5 hệ thống trên **cùng một corpus** và **cùng preprocessing**:

1. TF-IDF + K-means
2. NMF
3. LDA
4. Sentence Embedding + K-means
5. BERTopic

Sản phẩm cuối:
- biểu diễn 2D bằng PCA/SVD/UMAP;
- top words cho từng cụm/topic;
- tên topic do sinh viên đặt thủ công;
- tài liệu tiêu biểu;
- metric đánh giá phù hợp;
- phân tích tài liệu khó/topic khó diễn giải.

> **Nguyên tắc:** giữ cố định tập dữ liệu và preprocessing để so sánh công bằng. Representation có thể khác nhau theo yêu cầu thuật toán.


## Quy định bài làm

Các ô có `TODO` là phần sinh viên cần hoàn thiện.

Khuyến nghị:
- tối thiểu khoảng 1.000 tài liệu nếu tài nguyên cho phép.
- không dùng nhãn tham chiếu trong quá trình huấn luyện unsupervised.
- nếu có nhãn, chỉ dùng cho đánh giá ngoài.
- không kết luận chất lượng mô hình chỉ dựa trên một metric.


In [ ]:
# Nếu dùng Google Colab:
# %pip install -q pandas numpy scikit-learn matplotlib
# %pip install -q sentence-transformers bertopic umap-learn hdbscan gensim


In [ ]:
import re
import random
import unicodedata
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import NMF, LatentDirichletAllocation, TruncatedSVD, PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_colwidth", 180)


# 1. Nạp dữ liệu

CSV tối thiểu cần cột `text`. Có thể có thêm cột `label`.

### TODO 1
- sửa `DATA_PATH`;
- đọc dữ liệu;
- loại dòng rỗng;
- loại duplicate;
- báo cáo số document cuối.


In [ ]:
DATA_PATH = "data_vietnamese.csv"
TEXT_COL = "text"
LABEL_COL = None  # đổi thành "label" nếu có

# TODO: dùng dữ liệu thật
# df = pd.read_csv(DATA_PATH)

# Demo nhỏ để kiểm tra notebook
demo_data = [
    "Giá xăng trong nước tiếp tục tăng trong kỳ điều chỉnh mới.",
    "Ngân hàng công bố mức lãi suất tiết kiệm mới cho khách hàng.",
    "Thị trường chứng khoán tăng điểm sau phiên giao dịch tích cực.",
    "Đội tuyển Việt Nam giành chiến thắng trong trận giao hữu.",
    "Câu lạc bộ công bố huấn luyện viên mới trước mùa giải.",
    "Giải bóng đá quốc gia bước vào vòng đấu quyết định.",
    "Điện thoại mới có camera tốt nhưng thời lượng pin chưa ấn tượng.",
    "Khách hàng phản ánh giao hàng chậm và đóng gói chưa cẩn thận.",
    "Sản phẩm có chất lượng tốt, giao hàng nhanh và giá hợp lý.",
    "Trường đại học công bố chương trình tuyển sinh cho năm học mới.",
    "Sinh viên tham gia hội thảo về trí tuệ nhân tạo và dữ liệu.",
    "Bộ Giáo dục công bố thay đổi trong kế hoạch thi tốt nghiệp."
]
df = pd.DataFrame({TEXT_COL: demo_data})
df.head()


# 2. Preprocessing chung

Cố định một quy trình cho cả 5 hệ thống:
- Unicode normalization;
- lowercase;
- URL/email normalization;
- chuẩn hóa ký tự và khoảng trắng;
- có thể thêm tách từ tiếng Việt, nhưng phải áp dụng đồng nhất cho toàn bộ corpus.


In [ ]:
def preprocess_text(text):
    text = unicodedata.normalize("NFC", str(text))
    text = text.lower()
    text = re.sub(r"https?://\\S+|www\\.\\S+", " URL ", text)
    text = re.sub(r"\\b\\S+@\\S+\\.\\S+\\b", " EMAIL ", text)
    text = re.sub(r"[^0-9a-zA-ZÀ-ỹ_\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

n_before = len(df)
df = df.dropna(subset=[TEXT_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].astype(str)
df = df[df[TEXT_COL].str.strip().ne("")]
df = df.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)
df["text_clean"] = df[TEXT_COL].map(preprocess_text)
df["n_words"] = df["text_clean"].str.split().str.len()

print("Trước lọc:", n_before)
print("Sau lọc:", len(df))
df.head()


### TODO 2 — Khảo sát dữ liệu
Báo cáo:
- số document;
- mean/median/min/max số từ;
- số duplicate bị loại;
- 5 tài liệu ngắn nhất và dài nhất;
- nếu có `label`: phân bố label.


In [ ]:
summary = {
    "n_documents": len(df),
    "mean_words": df["n_words"].mean(),
    "median_words": df["n_words"].median(),
    "min_words": df["n_words"].min(),
    "max_words": df["n_words"].max(),
}
summary


# 3. Cố định corpus và số topic/cluster

Từ đây trở đi không lọc riêng document theo từng mô hình.

```python
docs = df["text_clean"].tolist()
```

Có thể tune nhiều giá trị K, nhưng phải chốt một `N_TOPICS` chung cho thí nghiệm so sánh chính.


In [ ]:
docs = df["text_clean"].tolist()
N_DOCS = len(docs)
N_TOPICS = 4 if N_DOCS < 100 else 8

print("N_DOCS =", N_DOCS)
print("N_TOPICS =", N_TOPICS)


# 4. Representation

## 4.1 TF-IDF
TF-IDF dùng cho K-means và NMF.

### TODO 3
Thử unigram và unigram+bigram, sau đó chọn một cấu hình và giải thích.


In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=10000,
    min_df=1 if N_DOCS < 100 else 3,
    max_df=0.95,
    ngram_range=(1, 2),
)
X_tfidf = tfidf_vectorizer.fit_transform(docs)

print("TF-IDF shape:", X_tfidf.shape)
print("Vocabulary:", len(tfidf_vectorizer.vocabulary_))


## 4.2 Count matrix cho LDA
LDA thường được fit trên raw counts thay vì TF-IDF.


In [ ]:
count_vectorizer = CountVectorizer(
    max_features=10000,
    min_df=1 if N_DOCS < 100 else 3,
    max_df=0.95,
    ngram_range=(1, 1),
)
X_count = count_vectorizer.fit_transform(docs)
print("Count shape:", X_count.shape)


# 5. Hệ 1 — TF-IDF + K-means

K-means gom document quanh các centroid.  
Dùng `k-means++`, `n_init=20`, `random_state=42`.


In [ ]:
kmeans_tfidf = KMeans(
    n_clusters=N_TOPICS,
    init="k-means++",
    n_init=20,
    random_state=RANDOM_STATE,
)
labels_km_tfidf = kmeans_tfidf.fit_predict(X_tfidf)
Counter(labels_km_tfidf)


In [ ]:
def top_terms_kmeans(model, vectorizer, n_words=10):
    terms = np.array(vectorizer.get_feature_names_out())
    rows = []
    for k, center in enumerate(model.cluster_centers_):
        ids = center.argsort()[::-1][:n_words]
        rows.append({
            "cluster": k,
            "top_words": ", ".join(terms[ids])
        })
    return pd.DataFrame(rows)

top_terms_kmeans(kmeans_tfidf, tfidf_vectorizer, 10)


### TODO 4 — K-means
- lấy ít nhất 3 document gần centroid nhất cho mỗi cluster;
- đặt tên cluster thủ công từ top words + representative docs;
- ghi lại cluster size.


In [ ]:
# TODO


# 6. Hệ 2 — NMF

NMF xấp xỉ ma trận document-term bằng hai ma trận không âm:

`X ≈ W @ H`

- `W[d,k]`: mức độ document d thuộc topic k;
- `H[k,v]`: mức độ từ v đại diện topic k.


In [ ]:
nmf = NMF(
    n_components=N_TOPICS,
    init="nndsvda",
    random_state=RANDOM_STATE,
    max_iter=500,
)
W_nmf = nmf.fit_transform(X_tfidf)
H_nmf = nmf.components_
labels_nmf = W_nmf.argmax(axis=1)

print(W_nmf.shape, H_nmf.shape)
Counter(labels_nmf)


In [ ]:
def top_terms_components(components, vectorizer, n_words=10, id_name="topic"):
    terms = np.array(vectorizer.get_feature_names_out())
    rows = []
    for k, comp in enumerate(components):
        ids = comp.argsort()[::-1][:n_words]
        rows.append({
            id_name: k,
            "top_words": ", ".join(terms[ids])
        })
    return pd.DataFrame(rows)

top_nmf = top_terms_components(H_nmf, tfidf_vectorizer, 10, "topic")
top_nmf


### TODO 5 — NMF
- lấy 3 document có `W[d,k]` lớn nhất cho mỗi topic;
- đặt tên topic thủ công;
- nhận xét topic có dễ diễn giải hơn K-means hay không.


In [ ]:
def representative_docs_nmf(W, raw_df, n_per_topic=3):
    rows = []
    for k in range(W.shape[1]):
        ids = np.argsort(W[:, k])[::-1][:n_per_topic]
        for rank, doc_id in enumerate(ids, 1):
            rows.append({
                "topic": k,
                "rank": rank,
                "weight": float(W[doc_id, k]),
                "text": raw_df.iloc[doc_id][TEXT_COL]
            })
    return pd.DataFrame(rows)

representative_docs_nmf(W_nmf, df)


# 7. Hệ 3 — LDA

LDA xem mỗi document là hỗn hợp nhiều topic và mỗi topic là phân phối trên vocabulary.

### Cần phân tích:
- document-topic distribution;
- topic-word distribution;
- perplexity;
- khả năng diễn giải.


In [ ]:
lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    learning_method="batch",
    max_iter=20,
    random_state=RANDOM_STATE,
)
doc_topic_lda = lda.fit_transform(X_count)
labels_lda = doc_topic_lda.argmax(axis=1)

print("Doc-topic:", doc_topic_lda.shape)
print("Perplexity:", lda.perplexity(X_count))
Counter(labels_lda)


In [ ]:
top_lda = top_terms_components(
    lda.components_,
    count_vectorizer,
    n_words=10,
    id_name="topic"
)
top_lda


### TODO 6 — LDA
- hiển thị phân phối topic của ít nhất 5 document;
- tìm document có 2 topic xác suất gần nhau;
- lấy 3 representative docs/topic;
- đặt tên topic thủ công;
- nhận xét mối quan hệ giữa perplexity và khả năng diễn giải.


In [ ]:
sample_n = min(5, len(df))
lda_probs = pd.DataFrame(
    doc_topic_lda[:sample_n],
    columns=[f"topic_{k}" for k in range(N_TOPICS)]
)
lda_probs.insert(0, "text", df.iloc[:sample_n][TEXT_COL].values)
lda_probs


# 8. Hệ 4 — Sentence Embedding + K-means

Gợi ý multilingual model:

`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`

### TODO 7
- encode đúng cùng `docs`;
- `normalize_embeddings=True`;
- fit K-means với cùng `N_TOPICS`;
- tính silhouette;
- lấy document gần centroid nhất;
- diễn giải cluster bằng top words lấy từ mean TF-IDF.


In [ ]:
# from sentence_transformers import SentenceTransformer
#
# embedding_model = SentenceTransformer(
#     "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
# )
#
# X_emb = embedding_model.encode(
#     docs,
#     show_progress_bar=True,
#     normalize_embeddings=True
# )
#
# kmeans_emb = KMeans(
#     n_clusters=N_TOPICS,
#     n_init=20,
#     random_state=RANDOM_STATE
# )
# labels_km_emb = kmeans_emb.fit_predict(X_emb)


In [ ]:
def top_words_by_cluster_from_tfidf(X_tfidf, labels, vectorizer, n_words=10):
    terms = np.array(vectorizer.get_feature_names_out())
    labels = np.asarray(labels)
    rows = []

    for k in sorted(set(labels)):
        ids = np.where(labels == k)[0]
        mean_vec = np.asarray(X_tfidf[ids].mean(axis=0)).ravel()
        best = mean_vec.argsort()[::-1][:n_words]
        rows.append({
            "cluster": int(k),
            "top_words": ", ".join(terms[best])
        })

    return pd.DataFrame(rows)

# Sau khi có labels_km_emb:
# top_words_by_cluster_from_tfidf(
#     X_tfidf, labels_km_emb, tfidf_vectorizer, 10
# )


# 9. Hệ 5 — BERTopic

Pipeline khái quát:

`Document → Sentence Embedding → UMAP → HDBSCAN → c-TF-IDF → Topic`

Lưu ý:
- BERTopic có thể tạo số topic khác `N_TOPICS`;
- topic `-1` thường là outlier;
- phải báo cáo số topic thực tế và outlier rate.


In [ ]:
# from bertopic import BERTopic
# from sentence_transformers import SentenceTransformer
#
# embedding_model = SentenceTransformer(
#     "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
# )
#
# topic_model = BERTopic(
#     embedding_model=embedding_model,
#     calculate_probabilities=True,
#     verbose=True
# )
#
# topics_bertopic, probs_bertopic = topic_model.fit_transform(docs)
# topic_model.get_topic_info().head(15)


### TODO 8 — BERTopic
- số topic không tính `-1`;
- outlier rate;
- top 10 words/topic;
- ít nhất 3 representative docs/topic;
- tên topic thủ công.


In [ ]:
# TODO


# 10. Trực quan hóa 2D

Yêu cầu:
- TF-IDF: Truncated SVD;
- sentence embedding: PCA hoặc UMAP;
- BERTopic: UMAP hoặc visualization tích hợp;
- màu theo cluster/topic assignment.

**Lưu ý:** 2D chỉ để trực quan hóa, không thay thế metric.


In [ ]:
svd_2d = TruncatedSVD(n_components=2, random_state=RANDOM_STATE)
X_tfidf_2d = svd_2d.fit_transform(X_tfidf)

def plot_2d(points, labels, title):
    plt.figure(figsize=(8, 6))
    labels = np.asarray(labels)
    for lab in sorted(set(labels)):
        ids = labels == lab
        plt.scatter(points[ids, 0], points[ids, 1], s=30, alpha=0.75, label=str(lab))
    plt.title(title)
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.legend(title="Cluster/Topic", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

plot_2d(X_tfidf_2d, labels_km_tfidf, "TF-IDF + K-means — SVD 2D")
plot_2d(X_tfidf_2d, labels_nmf, "NMF dominant topic — SVD 2D")
plot_2d(X_tfidf_2d, labels_lda, "LDA dominant topic — SVD 2D")


### TODO 9 — Embedding/BERTopic visualization
Ví dụ:
```python
pca = PCA(n_components=2, random_state=42)
X_emb_2d = pca.fit_transform(X_emb)
plot_2d(X_emb_2d, labels_km_emb, "Sentence Embedding + K-means")
```

Có thể thay PCA bằng UMAP và giải thích lựa chọn.


# 11. Đánh giá

## Silhouette
- gần 1: tách cụm tốt;
- gần 0: cụm chồng lấp;
- âm: nhiều điểm có thể phù hợp cluster khác hơn.

## Davies–Bouldin
- thấp hơn thường tốt hơn.

## LDA
- báo cáo perplexity;
- có thể bổ sung topic coherence.

## BERTopic/NMF
- nên kết hợp metric với đánh giá khả năng diễn giải.


In [ ]:
sil_km_tfidf = silhouette_score(
    X_tfidf,
    labels_km_tfidf,
    metric="cosine"
)

dbi_km_tfidf_2d = davies_bouldin_score(
    X_tfidf_2d,
    labels_km_tfidf
)

print("Silhouette TF-IDF + K-means:", sil_km_tfidf)
print("DBI trên SVD 2D:", dbi_km_tfidf_2d)
print("LDA perplexity:", lda.perplexity(X_count))


### TODO 10 — Topic coherence
Thực hiện ít nhất một trong hai:
- tính coherence cho NMF/LDA/BERTopic;
- hoặc xây rubric đánh giá thủ công khả năng diễn giải topic.

Không được kết luận “mô hình tốt nhất” chỉ từ perplexity hoặc silhouette.


# 12. Top words + tên topic + tài liệu tiêu biểu

Tạo bảng:

| System | Topic/Cluster | Top words | Tên topic thủ công | Representative docs |
|---|---:|---|---|---|
| TF-IDF + K-means | 0 | ... | ... | ... |
| NMF | 0 | ... | ... | ... |
| LDA | 0 | ... | ... | ... |
| Embedding + K-means | 0 | ... | ... | ... |
| BERTopic | 0 | ... | ... | ... |

Tên topic phải dựa trên **cả top words lẫn representative docs**.


In [ ]:
manual_names_km_tfidf = {}
manual_names_nmf = {}
manual_names_lda = {}
manual_names_km_emb = {}
manual_names_bertopic = {}


# 13. Bảng so sánh 5 hệ thống

| System | Representation | # topic/cluster | Silhouette | Perplexity/Coherence | Outlier rate | Diễn giải |
|---|---|---:|---:|---:|---:|---|
| TF-IDF + K-means | TF-IDF | | | N/A | 0 | |
| NMF | TF-IDF | | | | 0 | |
| LDA | Count | | N/A | | 0 | |
| Embedding + K-means | Sentence Embedding | | | N/A | 0 | |
| BERTopic | Embedding + UMAP + HDBSCAN | | | | | |


In [ ]:
results = pd.DataFrame([
    {
        "system": "TF-IDF + K-means",
        "representation": "TF-IDF",
        "n_topics": len(set(labels_km_tfidf)),
        "silhouette": sil_km_tfidf,
        "perplexity": np.nan,
        "outlier_rate": 0.0,
    },
    {
        "system": "NMF",
        "representation": "TF-IDF",
        "n_topics": N_TOPICS,
        "silhouette": np.nan,
        "perplexity": np.nan,
        "outlier_rate": 0.0,
    },
    {
        "system": "LDA",
        "representation": "Count",
        "n_topics": N_TOPICS,
        "silhouette": np.nan,
        "perplexity": lda.perplexity(X_count),
        "outlier_rate": 0.0,
    },
    # TODO: thêm Sentence Embedding + K-means
    # TODO: thêm BERTopic
])
results


# 14. Error analysis — tài liệu khó

Chọn ít nhất **10 document** thuộc một hoặc nhiều nhóm:
- xa centroid;
- topic distribution gần đều;
- BERTopic gán `-1`;
- TF-IDF và embedding gán sang hai cụm khác nhau rõ rệt;
- document đa chủ đề;
- document quá ngắn;
- document chứa nhiều từ chung chung.

Tạo bảng:

| Document | K-means TF-IDF | NMF | LDA | Embedding K-means | BERTopic | Nhận xét |
|---|---:|---:|---:|---:|---:|---|


### TODO 11 — Câu hỏi phân tích

1. Hệ thống nào tạo topic dễ diễn giải nhất?
2. Hệ thống nào nhạy với từ khóa bề mặt nhất?
3. Embedding xử lý paraphrase tốt hơn TF-IDF ở ví dụ nào?
4. LDA có document nào là hỗn hợp nhiều topic rõ rệt không?
5. BERTopic có bao nhiêu outlier và các outlier có đặc điểm gì?
6. Topic nào khó đặt tên nhất? Vì sao?
7. Có cluster/topic nào trộn hai chủ đề không?
8. Metric định lượng có đồng thuận với đánh giá thủ công không?


# 15. Chọn số cluster/topic

### TODO 12
Thử ít nhất:
```text
K ∈ {4, 6, 8, 10}
```

Với K-means:
- silhouette;
- cluster size;
- khả năng diễn giải.

Với NMF/LDA:
- coherence;
- perplexity;
- khả năng diễn giải.

Giải thích vì sao chọn `N_TOPICS` cuối cùng.


In [ ]:
# TODO skeleton
#
# rows = []
# for k in [4, 6, 8, 10]:
#     model = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
#     labels = model.fit_predict(X_tfidf)
#     sil = silhouette_score(X_tfidf, labels, metric="cosine")
#     rows.append({"k": k, "silhouette": sil})
#
# pd.DataFrame(rows)


# 16. Cluster stability

### TODO 13
Chạy K-means với:
```text
random_state = 1, 2, 3, 4, 5
```

Theo dõi:
- silhouette;
- cluster size;
- nếu đã học ARI/NMI: so assignment giữa các lần chạy.

Kết luận: kết quả có ổn định hay phụ thuộc mạnh vào seed?


In [ ]:
# TODO


# 17. Kết luận cuối

Viết khoảng **300–500 từ**:
- ưu/nhược điểm TF-IDF + K-means;
- NMF và LDA khác nhau ra sao;
- sentence embedding thay đổi cấu trúc cluster như thế nào;
- BERTopic có lợi thế gì;
- metric định lượng và đánh giá thủ công có đồng thuận không;
- hệ thống nào phù hợp nhất với dữ liệu và vì sao.


# 18. Checklist trước khi nộp

- [ ] Dùng dữ liệu tiếng Việt thật.
- [ ] Loại duplicate và document rỗng.
- [ ] Cố định cùng `docs` cho 5 hệ thống.
- [ ] Cố định preprocessing.
- [ ] Có TF-IDF + K-means.
- [ ] Có NMF.
- [ ] Có LDA.
- [ ] Có Sentence Embedding + K-means.
- [ ] Có BERTopic.
- [ ] Có biểu diễn 2D.
- [ ] Có top words.
- [ ] Có tên topic thủ công.
- [ ] Có ít nhất 3 representative docs/topic.
- [ ] Có metric định lượng phù hợp.
- [ ] Có ít nhất 10 trường hợp khó.
- [ ] Có thí nghiệm chọn số topic/cluster.
- [ ] Có nhận xét stability.
- [ ] Có kết luận cuối.


# 19. Thang điểm

| Hạng mục | Điểm |
|---|---:|
| Dữ liệu + preprocessing | 1.0 |
| TF-IDF + K-means | 1.0 |
| NMF | 1.0 |
| LDA | 1.0 |
| Sentence Embedding + K-means | 1.0 |
| BERTopic | 1.0 |
| Trực quan hóa 2D | 1.0 |
| Top words + topic naming + representative docs | 1.0 |
| Đánh giá định lượng | 1.0 |
| Error analysis + kết luận | 1.0 |
| **Tổng** | **10.0** |
